In [110]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [111]:
DATA = "../data/raw/listings.csv"
REPORT = "../reports/"
CLEANED = "../data/processed/cleanedV1.csv"

In [112]:
df = pd.read_csv(DATA)
init_count = len(df)

# Cleaning


## Price


In [ ]:
def tag_price(df):
    df = df.copy()
    df["price_tag"] = ""

    fc = [r"\$", "dollars", "dollar", "usd"]
    sale = [
        "hostel",
        "sale",
        "sold",
        "for sell",
        "sell",
        "for sale",
        "buy",
        "buyer",
        "seller",
        "airbnb",  
        "air bnb",
    ]
    period = [
        "per night",
        "night",
        "per day",
        "per year",
        "quaterly",
        "yearly",
        "annually",
        "airbnb",  
        "air bnb",
        "daily",
    ]

    sale_pattern = "|".join(sale)
    fc_pattern = "|".join(fc)
    period_pattern = "|".join(period)

    has_fc = df["description"].str.contains(fc_pattern, case=False, na=False)
    has_sale = df["description"].str.contains(sale_pattern, case=False, na=False)
    has_period = df["description"].str.contains(period_pattern, case=False, na=False)

    # Apply tags in order of priority (most specific last, or use elif logic)
    # Option 1: If you want sale OR period to both be removed, apply them first

    df.loc[has_sale, "price_tag"] = "sale"
    df.loc[has_period, "price_tag"] = "period"

    # Then apply foreign currency only if not already tagged
    fc_condition = has_fc & ~has_sale & ~has_period
    df.loc[fc_condition, "price_tag"] = "foreign"

    print(f"Sale: {has_sale.sum()} listings")
    print(f"Period: {has_period.sum()} listings")
    print(f"Foreign: {fc_condition.sum()} listings")

    return df


df = tag_price(df)

Sale: 442 listings
Period: 301 listings
Foreign: 3667 listings


In [114]:
def remove_bad_listings(df):
    """Removes wrong listings based on specified period of stay (daily,quaterly etc), sale quote leakages"""
    df.copy()

    df = df[df["price_tag"] != "sale"]
    df = df[df["price_tag"] != "period"]

    df = df[~((df["price"] > 15000) & (df["price_tag"] == "foreign"))]

    return df


df = remove_bad_listings(df)


In [115]:
df[df["price"] < 500]

,Unnamed: 0,url,description,fetch_date,title,location,house_type,bathrooms,bedrooms,price,...,Kitchen Shelf,Microwave,Pop Ceiling,Pre-Paid Meter,Refrigerator,TV,Tiled Floor,Wardrobe,Wi-Fi,price_tag
1455,1455,https://jiji.com.gh/dansoman/houses-apartments...,Single room at exhibition round about viewing ...,2025-12-26 21:14:28.330890,1bdrm Apartment in Dansoman Exhibition for rent,Dansoman,Apartment,1,1,400.0,...,0,0,0,0,0,0,0,0,0,
1766,1766,https://jiji.com.gh/mamprobi/houses-apartments...,Big single room and porch wall gated house vie...,2025-12-26 21:14:22.787060,1bdrm Apartment in Mamprobi Post Office for rent,Mamprobi,Apartment,1,1,400.0,...,0,0,0,0,0,0,0,0,0,
3648,3648,https://jiji.com.gh/tema-metropolitan/houses-a...,Single room self contained up for rent at Gbet...,2025-12-26 21:16:10.048436,"1bdrm Apartment in Gbetsile U.S.A, Tema Metrop...",Tema Metropolitan,Apartment,1,1,250.0,...,0,0,0,0,0,0,0,0,0,
3670,3670,https://jiji.com.gh/police-station-oyibi/house...,Single room self-contained for rent at Malejor...,2025-12-26 21:16:10.042796,"1bdrm Apartment in Abventures, Police Station ...",Oyibi,Apartment,1,1,400.0,...,0,0,0,0,0,0,0,0,0,
3708,3708,https://jiji.com.gh/tema-metropolitan/houses-a...,Spacious Single room apartment up for rent at ...,2025-12-26 21:16:09.963095,1bdrm Apartment in Gbetsile Bumper Tema Metrop...,Tema Metropolitan,Apartment,1,1,250.0,...,0,0,0,0,0,0,0,0,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14160,14161,https://jiji.com.gh/ga-east-municipal/houses-a...,Single room for rent at madina social welfare ...,2025-12-26 21:12:41.015143,"1bdrm House in Madina Social, Ga East Municipa...",Ga East Municipal,House,1,1,200.0,...,0,0,0,0,0,0,0,0,0,
14175,14176,https://jiji.com.gh/adam-nana/houses-apartment...,Very nice single room self contain for rent at...,2025-12-26 21:12:41.010966,"1bdrm Apartment in Nurses Quarters, Adam Nana ...",Kasoa,Apartment,1,1,300.0,...,0,0,0,0,0,0,0,0,0,
14215,14216,https://jiji.com.gh/ga-west-municipal/houses-a...,Chamber self-contain for rent at amasaman Asal...,2025-12-26 21:12:39.684061,"1bdrm Apartment in Amasaman Asalahaja, Ga West...",Ga West Municipal,Apartment,1,1,400.0,...,0,0,0,0,0,0,0,0,0,
14533,14534,https://jiji.com.gh/ashaiman-municipal/houses-...,Chamber and Hall with Porch for Rent – Ataadek...,2025-12-26 21:12:34.719244,"1bdrm Room & Parlour in Kasapreko, Ashaiman Mu...",Ashaiman Municipal,Room & Parlour,1,1,350.0,...,0,0,0,0,0,0,0,0,0,


In [116]:
df[df["price"] > 15000]

,Unnamed: 0,url,description,fetch_date,title,location,house_type,bathrooms,bedrooms,price,...,Kitchen Shelf,Microwave,Pop Ceiling,Pre-Paid Meter,Refrigerator,TV,Tiled Floor,Wardrobe,Wi-Fi,price_tag
4,4,https://jiji.com.gh/abelemkpe/houses-apartment...,*TO LET* \n\n *5-BEDROOM LUXURY HOME* \n\n*Loc...,2025-12-26 21:14:51.668787,5bdrm House in Abelemkpe for rent,Abelemkpe,House,5,5,48000.0,...,0,0,0,0,0,0,0,0,0,
45,45,https://jiji.com.gh/tema-metropolitan/houses-a...,A 4 bedroom 4 bathroom fully furnished house w...,2025-12-26 21:14:50.648510,Furnished 4bdrm House in Tema Metropolitan for...,Tema Metropolitan,House,4,4,18607.0,...,0,0,0,0,0,0,0,0,0,
121,121,https://jiji.com.gh/tema-metropolitan/houses-a...,3bedroom furnished house for rent around the j...,2025-12-26 21:14:50.551791,"Furnished 3bdrm House in Bronko Housing, Tema ...",Tema Metropolitan,House,3,3,30000.0,...,0,0,0,0,0,1,0,0,0,
131,131,https://jiji.com.gh/ability/houses-apartments-...,"4bedroom apartment for rent in east legon, goi...",2025-12-26 21:14:49.439146,Furnished 4bdrm Apartment in Ability Square Do...,Adjiriganor,Apartment,4,4,15606.0,...,0,0,0,0,0,0,0,0,0,
157,157,https://jiji.com.gh/adjiriganor/houses-apartme...,3bedrooms apartment furnished at adjiringanor ...,2025-12-26 21:14:48.754007,"Furnished 3bdrm Apartment in Beijing, Adjiriga...",Adjiriganor,Apartment,3,3,18480.0,...,0,0,0,0,0,1,0,0,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14731,14732,https://jiji.com.gh/west-hills-mall/houses-apa...,Fully furnished 2bedroom townhouse for rent. 2...,2025-12-26 21:12:31.723035,Furnished 2bdrm Townhouse / Terrace in Westhil...,Weija,Townhouse / Terrace,2,2,30000.0,...,0,0,0,0,0,1,0,0,0,
14733,14734,https://jiji.com.gh/tesano/houses-apartments-f...,3 Bedrooms self compound house for rent at Tes...,2025-12-26 21:12:31.721968,3bdrm House in Tesano for rent,Tesano,House,2,3,16000.0,...,0,0,0,0,0,0,0,0,0,
14753,14754,https://jiji.com.gh/weija/houses-apartments-fo...,21 Bedrooms newly built house for rent at Weij...,2025-12-26 21:12:31.394917,20bdrm House in Weija for rent,Weija,House,19,20,28000.0,...,0,0,0,0,0,0,0,0,0,
14755,14756,https://jiji.com.gh/east-legon/houses-apartmen...,Now renting at east legon American house area ...,2025-12-26 21:12:31.394127,4bdrm Duplex in East Legon for rent,East Legon,Duplex,4,4,17000.0,...,0,0,0,0,0,0,0,0,0,


In [117]:
df = df[~((df["bedrooms"] < 4) & (df["price"] > 50000))]
df = df[~((df["bedrooms"] < 4) & (df["price"] >= 40000))]
df = df[~((df["bedrooms"] < 3) & (df["price"] >= 30000))]

df = df[~(df["price"] > 50000)]

In [118]:
df

,Unnamed: 0,url,description,fetch_date,title,location,house_type,bathrooms,bedrooms,price,...,Kitchen Shelf,Microwave,Pop Ceiling,Pre-Paid Meter,Refrigerator,TV,Tiled Floor,Wardrobe,Wi-Fi,price_tag
2,2,https://jiji.com.gh/ledzokuku-krowor/houses-ap...,Wall and gated spacious single self contained ...,2025-12-26 21:14:51.677264,"1bdrm Apartment in Teshie Nungua, Ledzokuku-Kr...",Ledzokuku-Krowor,Apartment,1,1,1200.0,...,0,0,0,0,0,0,0,0,0,
3,3,https://jiji.com.gh/east-legon/houses-apartmen...,Newly built Executive chamber and hall self ap...,2025-12-26 21:14:51.674819,1bdrm Room & Parlour in East Legon for rent,East Legon,Room & Parlour,1,1,4000.0,...,0,0,0,0,0,0,0,0,0,
4,4,https://jiji.com.gh/abelemkpe/houses-apartment...,*TO LET* \n\n *5-BEDROOM LUXURY HOME* \n\n*Loc...,2025-12-26 21:14:51.668787,5bdrm House in Abelemkpe for rent,Abelemkpe,House,5,5,48000.0,...,0,0,0,0,0,0,0,0,0,
5,5,https://jiji.com.gh/pillar-2/houses-apartments...,Newly Built Single Room Self Contained For Ren...,2025-12-26 21:14:51.667797,1bdrm Apartment in Francis Quarcoe Pillar 2 fo...,Dome,Apartment,1,1,1200.0,...,0,0,0,0,0,0,0,0,0,
7,7,https://jiji.com.gh/dome/houses-apartments-for...,3 Bedroom Self Compound House For Rent At Dome...,2025-12-26 21:14:51.663040,"3bdrm House in Francis Quarcoe, Dome for rent",Dome,House,3,3,8000.0,...,0,0,0,0,0,0,0,0,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14863,14864,https://jiji.com.gh/old-barrier/houses-apartme...,Newly built executive 2bedroom apartment for r...,2025-12-26 21:12:30.235215,2bdrm Apartment in Old Barrier for rent,Weija,Apartment,2,2,2500.0,...,0,0,0,0,0,0,0,0,0,
14864,14865,https://jiji.com.gh/haatso/houses-apartments-f...,Newly built 2 bedrooms apartment for rent at h...,2025-12-26 21:12:30.234140,"2bdrm Apartment in Max Properties, Haatso for ...",Haatso,Apartment,3,2,3700.0,...,0,0,0,0,0,0,0,0,0,
14865,14866,https://jiji.com.gh/tantra-hills/houses-apartm...,Chamber and Hall self contained for rent at Ta...,2025-12-26 21:12:30.234695,"1bdrm Apartment in Madev Realty, Tantra Hills ...",Tantra Hills,Apartment,1,1,1300.0,...,0,0,0,0,0,0,0,0,0,
14866,14867,https://jiji.com.gh/west-legon/houses-apartmen...,2 bedrooms\nwashroom=3\ninstalled Ac\ninbuilt ...,2025-12-26 21:12:30.232797,2bdrm Apartment in West Legon for rent,West Legon,Apartment,3,2,6500.0,...,0,0,0,0,0,0,0,0,0,


In [119]:
# Price vs Bedrooms
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(df["bedrooms"], df["price"], alpha=0.3, s=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel("Bedrooms")
axes[0].set_ylabel("Price")
axes[0].set_title("Price vs Bedrooms (Linear Scale)")

v_prices = df[df["price"] > 0]
axes[1].scatter(v_prices["bedrooms"], v_prices["price"], alpha=0.3, s=10)
axes[1].set_xlabel("Bedrooms")
axes[1].set_ylabel("Price (log scale)")
axes[1].set_yscale("log")
axes[1].set_title("Price vs Bedrooms (Log Scale)")
axes[1].grid(True, alpha=0.3)

price_quantiles = df["price"].quantile([0.1, 0.25, 0.5, 0.75, 0.9])
for q in price_quantiles:
    axes[0].axhline(q, color="red", alpha=0.2, linestyle="--")
    axes[1].axhline(q, color="red", alpha=0.2, linestyle="--")

plt.tight_layout()
plt.savefig(f"{REPORT}01_price_bedrooms.png", dpi=150)
plt.close()
print("✓ Saved price vs bedrooms inspection")


✓ Saved price vs bedrooms inspection


In [120]:
plt.figure(figsize=(12, 6))
df.boxplot(column="price", by="house_type", ax=plt.gca())
plt.yscale("log")
plt.xticks(rotation=45, ha="right")
plt.title("Price Distribution by Property Type (Log Scale)")
plt.suptitle("")
plt.ylabel("Price (log)")
plt.tight_layout()
plt.savefig(f"{REPORT}01_price_vs_property_type.png", dpi=150)
plt.close()
print("✓ Saved price vs property type inspection")

✓ Saved price vs property type inspection


In [121]:
top_neighborhoods = df["location"].value_counts().head(30).index
df_top = df[df["location"].isin(top_neighborhoods)]

plt.figure(figsize=(14, 8))
df_top.boxplot(column="price", by="location", ax=plt.gca())
plt.yscale("log")
plt.xticks(rotation=45, ha="right")
plt.title("Price Distribution by Neighborhood (Top 20, Log Scale)")
plt.suptitle("")
plt.ylabel("Price (log)")
plt.tight_layout()
plt.savefig(f"{REPORT}01_price_vs_neighborhood.png", dpi=150)
plt.close()
print("✓ Saved price vs neighborhood inspection")

✓ Saved price vs neighborhood inspection


In [122]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Overall histogram
axes[0, 0].hist(df["price"], bins=100, edgecolor="black", alpha=0.7)
axes[0, 0].set_xlabel("Price")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].set_title("Price Distribution - Looking for Spikes")
axes[0, 0].axvline(df["price"].median(), color="red", label="Median", linestyle="--")
axes[0, 0].legend()

# Log histogram
axes[0, 1].hist(
    np.log10(df[df["price"] > 0]["price"]), bins=100, edgecolor="black", alpha=0.7
)
axes[0, 1].set_xlabel("Log10(Price)")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].set_title("Log Price Distribution - Looking for Modes")

# Suspicious price values
price_counts = df["price"].value_counts().head(20)
axes[1, 0].barh(range(len(price_counts)), price_counts.values)
axes[1, 0].set_yticks(range(len(price_counts)))
axes[1, 0].set_yticklabels([f"${p:,.0f}" for p in price_counts.index])
axes[1, 0].set_xlabel("Count")
axes[1, 0].set_title("Top 20 Most Common Exact Prices (Suspicious?)")

# Price percentiles
percentiles = range(0, 101, 5)
price_percentiles = [df["price"].quantile(p / 100) for p in percentiles]
axes[1, 1].plot(percentiles, price_percentiles, marker="o")
axes[1, 1].set_xlabel("Percentile")
axes[1, 1].set_ylabel("Price")
axes[1, 1].set_title("Price Percentile Curve - Looking for Jumps")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{REPORT}01_price_patterns.png", dpi=150)
plt.close()
print("✓ Saved detailed price pattern inspection")


✓ Saved detailed price pattern inspection


In [123]:
print("\n--- PRICE STATISTICS FOR THRESHOLD DECISIONS ---")
print(f"Minimum price: ${df['price'].min():,.2f}")
print(f"1st percentile: ${df['price'].quantile(0.01):,.2f}")
print(f"5th percentile: ${df['price'].quantile(0.05):,.2f}")
print(f"25th percentile: ${df['price'].quantile(0.25):,.2f}")
print(f"Median: ${df['price'].quantile(0.50):,.2f}")
print(f"75th percentile: ${df['price'].quantile(0.75):,.2f}")
print(f"95th percentile: ${df['price'].quantile(0.95):,.2f}")
print(f"99th percentile: ${df['price'].quantile(0.99):,.2f}")
print(f"Maximum price: ${df['price'].max():,.2f}")



--- PRICE STATISTICS FOR THRESHOLD DECISIONS ---
Minimum price: $175.00
1st percentile: $500.00
5th percentile: $800.00
25th percentile: $2,000.00
Median: $3,500.00
75th percentile: $7,500.00
95th percentile: $15,000.00
99th percentile: $32,000.00
Maximum price: $50,000.00


In [124]:
suspicious_values = [1, 10, 100, 1000, 9999, 99999]
for val in suspicious_values:
    count = (df["price"] == val).sum()
    if count > 0:
        print(f"  ⚠ Exact price = ${val}: {count} listings")


  ⚠ Exact price = $1000: 268 listings
  ⚠ Exact price = $9999: 3 listings


In [125]:
prices = df["price"].dropna()

# Raw price stats
price_stats = {
    "count": len(prices),
    "mean": prices.mean(),
    "median": prices.median(),
    "std": prices.std(),
    "min": prices.min(),
    "max": prices.max(),
    "q25": prices.quantile(0.25),
    "q75": prices.quantile(0.75),
    "iqr": prices.quantile(0.75) - prices.quantile(0.25),
}

print("\nRaw Price Statistics:")
for k, v in price_stats.items():
    print(f"  {k:10s}: {v:,.2f}" if isinstance(v, float) else f"  {k:10s}: {v:,}")

# Log price statistics
log_prices = np.log(prices[prices > 0])
log_stats = {
    "count": len(log_prices),
    "mean": log_prices.mean(),
    "median": log_prices.median(),
    "std": log_prices.std(),
}

print("\nLog Price Statistics:")
for k, v in log_stats.items():
    print(f"  {k:10s}: {v:,.4f}" if isinstance(v, float) else f"  {k:10s}: {v:,}")



Raw Price Statistics:
  count     : 11,551
  mean      : 5,672.13
  median    : 3,500.00
  std       : 5,960.02
  min       : 175.00
  max       : 50,000.00
  q25       : 2,000.00
  q75       : 7,500.00
  iqr       : 5,500.00

Log Price Statistics:
  count     : 11,551
  mean      : 8.2242
  median    : 8.1605
  std       : 0.9236


In [126]:
df[df["price"] > 25000].groupby("location")["price"].count().sort_values(
    ascending=False
)


location
East Legon                  55
Airport Residential Area    25
Cantonments                 15
Accra Metropolitan          14
Teshie                      13
Spintex                     11
Tema Metropolitan           10
Dzorwulu                     8
Labone                       5
Abelemkpe                    4
West Legon                   4
Roman Ridge                  4
Adjiriganor                  4
Ashaley Botwe                3
Dworwulu                     2
Oyarifa                      2
Achimota                     1
Akweteyman                   1
Burma Camp                   1
Awoshie                      1
Ashaiman Municipal           1
Greater Accra                1
North Legon                  1
Labadi                       1
Osu                          1
Weija                        1
Name: price, dtype: int64

In [127]:
df.groupby("location")["price"].median().sort_values()


location
Nima                350.0
Banana Inn          750.0
Santa Maria        1000.0
Mamprobi           1075.0
Nii Boi Town       1200.0
                   ...   
South Shiashie    12270.0
Okponglo          12950.0
Labone            15000.0
Ridge             18800.0
Cantonments       23500.0
Name: price, Length: 84, dtype: float64

In [128]:
df.to_csv(CLEANED)